In [1]:
import pandas as pd

In [2]:
from sklearn.cluster import KMeans

In [3]:
from sklearn.preprocessing import StandardScaler


### LOAD DATA

In [4]:
orders = pd.read_excel('orders.xlsx')
order_items = pd.read_excel('order_items.xlsx')
customers = pd.read_excel('Customers.xlsx')

In [5]:
orders.head()


,order_id,customer_id,Device_Type,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,logistics_company_id
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,Web,delivered,2017-10-02 10:56:00,2017-10-02 11:07:00,2017-10-04 19:55:00,2017-10-10 21:25:00,2017-10-18,5
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,Web,delivered,2018-07-24 20:41:00,2018-07-26 03:24:00,2018-07-26 14:31:00,2018-08-07 15:27:00,2018-08-13,3
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,Web,delivered,2018-08-08 08:38:00,2018-08-08 08:55:00,2018-08-08 13:50:00,2018-08-17 18:06:00,2018-09-04,3
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,Web,delivered,2017-11-18 19:28:00,2017-11-18 19:45:00,2017-11-22 13:39:00,2017-12-02 00:28:00,2017-12-15,5
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,Web,delivered,2018-02-13 21:18:00,2018-02-13 22:20:00,2018-02-14 19:46:00,2018-02-16 18:17:00,2018-02-26,5


In [6]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 10 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   Device_Type                    99441 non-null  object        
 3   order_status                   99441 non-null  object        
 4   order_purchase_timestamp       99441 non-null  datetime64[ns]
 5   order_approved_at              99281 non-null  datetime64[ns]
 6   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 7   order_delivered_customer_date  96476 non-null  datetime64[ns]
 8   order_estimated_delivery_date  99441 non-null  datetime64[ns]
 9   logistics_company_id           99441 non-null  int64         
dtypes: datetime64[ns](5), int64(1), object(4)
memory usage: 7.6+ MB


In [7]:
orders.isnull().sum()

order_id                            0
customer_id                         0
Device_Type                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
logistics_company_id                0
dtype: int64

In [8]:
orders[(orders['order_status'] == 'delivered') & (orders['order_delivered_customer_date'].isna())]


,order_id,customer_id,Device_Type,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,logistics_company_id
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,Web,delivered,2017-11-28 17:44:00,2017-11-28 17:56:00,2017-11-30 18:12:00,NaT,2017-12-18,5
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,Web,delivered,2018-06-20 06:58:00,2018-06-20 07:19:00,2018-06-25 08:05:00,NaT,2018-07-16,5
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,Web,delivered,2018-07-01 17:05:00,2018-07-01 17:15:00,2018-07-03 13:57:00,NaT,2018-07-30,5
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,Web,delivered,2018-07-01 22:05:00,2018-07-01 22:15:00,2018-07-03 13:57:00,NaT,2018-07-30,5
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,Web,delivered,2018-07-01 21:14:00,2018-07-01 21:29:00,2018-07-03 09:28:00,NaT,2018-07-24,5
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,Mobile,delivered,2017-05-25 23:22:00,2017-05-25 23:30:00,NaT,NaT,2017-06-23,4
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,Web,delivered,2018-06-08 12:09:00,2018-06-08 12:36:00,2018-06-12 14:10:00,NaT,2018-06-26,5
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,Web,delivered,2018-06-27 16:09:00,2018-06-27 16:29:00,2018-07-03 19:26:00,NaT,2018-07-19,5


In [9]:
orders[(orders['order_status'] == 'delivered') & (orders['order_delivered_carrier_date'].isna())]


,order_id,customer_id,Device_Type,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,logistics_company_id
73222,2aa91108853cecb43c84a5dc5b277475,afeb16c7f46396c0ed54acb45ccaaa40,Web,delivered,2017-09-29 08:52:00,2017-09-29 09:07:00,NaT,2017-11-20 19:44:00,2017-11-14,5
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,Mobile,delivered,2017-05-25 23:22:00,2017-05-25 23:30:00,NaT,NaT,2017-06-23,4


In [10]:
orders.dropna(subset=['order_approved_at'], inplace=True)
orders.dropna(subset=['order_delivered_carrier_date'], inplace=True)
orders.dropna(subset=['order_delivered_customer_date'], inplace=True)


In [11]:
orders.isnull().sum()

order_id                         0
customer_id                      0
Device_Type                      0
order_status                     0
order_purchase_timestamp         0
order_approved_at                0
order_delivered_carrier_date     0
order_delivered_customer_date    0
order_estimated_delivery_date    0
logistics_company_id             0
dtype: int64

In [12]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,Gender,Customer_Login_type,Age
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,Female,Member,28
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,Female,Member,22
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,Female,Member,35
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,Female,Member,18
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,Female,Member,21


In [13]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 6 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   Gender                    99441 non-null  object
 4   Customer_Login_type       99441 non-null  object
 5   Age                       99441 non-null  int64 
dtypes: int64(2), object(4)
memory usage: 4.6+ MB


In [14]:
customers.isnull().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
Gender                      0
Customer_Login_type         0
Age                         0
dtype: int64

In [15]:
order_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  object        
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  object        
 3   seller_id            112650 non-null  object        
 4   shipping_limit_date  112650 non-null  datetime64[ns]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(3)
memory usage: 6.0+ MB


In [16]:
order_items .isnull().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [17]:
orders = orders[orders['order_status'] == 'delivered']


In [18]:
# دمج الطلبات مع العملاء
df = orders.merge(customers, on='customer_id', how='left')
df

,order_id,customer_id,Device_Type,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,logistics_company_id,customer_unique_id,customer_zip_code_prefix,Gender,Customer_Login_type,Age
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,Web,delivered,2017-10-02 10:56:00,2017-10-02 11:07:00,2017-10-04 19:55:00,2017-10-10 21:25:00,2017-10-18,5,7c396fd4830fd04220f754e42b4e5bff,3149,Male,Member,39
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,Web,delivered,2018-07-24 20:41:00,2018-07-26 03:24:00,2018-07-26 14:31:00,2018-08-07 15:27:00,2018-08-13,3,af07308b275d755c9edb36a90c618231,47813,Female,Member,56
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,Web,delivered,2018-08-08 08:38:00,2018-08-08 08:55:00,2018-08-08 13:50:00,2018-08-17 18:06:00,2018-09-04,3,3a653a41f6f9fc3d2a113cf8398680e8,75265,Male,Member,62
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,Web,delivered,2017-11-18 19:28:00,2017-11-18 19:45:00,2017-11-22 13:39:00,2017-12-02 00:28:00,2017-12-15,5,7c142cf63193a1473d2e66489a9ae977,59296,Female,Member,28
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,Web,delivered,2018-02-13 21:18:00,2018-02-13 22:20:00,2018-02-14 19:46:00,2018-02-16 18:17:00,2018-02-26,5,72632f0f9dd73dfee390c9b22eb56dd6,9195,Male,Member,52
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96450,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,Web,delivered,2017-03-09 09:54:00,2017-03-09 09:54:00,2017-03-10 11:18:00,2017-03-17 15:08:00,2017-03-28,5,6359f309b166b0196dbf7ad2ac62bb5a,12209,Male,Member,60
96451,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,Web,delivered,2018-02-06 12:58:00,2018-02-06 13:10:00,2018-02-07 23:22:00,2018-02-28 17:37:00,2018-03-02,5,da62f9e57a76d978d02ab5362c509660,11722,Female,Member,54
96452,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,Web,delivered,2017-08-27 14:46:00,2017-08-27 15:04:00,2017-08-28 20:52:00,2017-09-21 11:24:00,2017-09-27,3,737520a9aad80b3fbbdad19b66b37b30,45920,Male,Member,41
96453,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,Web,delivered,2018-01-08 21:28:00,2018-01-08 21:36:00,2018-01-12 15:35:00,2018-01-25 23:32:00,2018-02-15,1,5097a5312c8b157bb7be58ae360ef43c,28685,Male,Member,36


In [19]:
# حساب عدد الطلبات لكل عميل
num_orders = df.groupby('customer_unique_id')['order_id'].nunique().reset_index(name='num_orders')

In [20]:
# حساب المبلغ الإجمالي لكل عميل
merged = df.merge(order_items, on='order_id', how='left')
merged['total_price'] = merged['price'] + merged['freight_value']
total_spent = merged.groupby('customer_unique_id')['total_price'].sum().reset_index(name='total_spent')


In [21]:
# حساب متوسط قيمة الطلب
avg_order_value = total_spent.merge(num_orders, on='customer_unique_id')
avg_order_value['avg_order_value'] = avg_order_value['total_spent'] / avg_order_value['num_orders']

In [22]:
# اختيار الخصائص للتصنيف
features = avg_order_value[['num_orders', 'total_spent', 'avg_order_value']]

In [23]:
# توحيد القيم
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

In [24]:
# تطبيق KMeans
kmeans = KMeans(n_clusters=4, random_state=42,n_init=10)
avg_order_value['customer_segment'] = kmeans.fit_predict(scaled_features)

In [25]:
from sklearn.metrics import silhouette_score

segment_names = {
    0: 'Occasional',
    1: 'High Value',
    2: 'Low Spender',
    3: 'Frequent Buyer'

}

In [26]:
avg_order_value['segment_name'] = avg_order_value['customer_segment'].map(segment_names)


In [27]:
score = silhouette_score(scaled_features, avg_order_value['customer_segment'])
print(" Silhouette Score:", round(score, 3))


 Silhouette Score: 0.754


In [28]:
print(avg_order_value.columns)


Index(['customer_unique_id', 'total_spent', 'num_orders', 'avg_order_value',
       'customer_segment', 'segment_name'],
      dtype='object')


In [29]:
avg_order_value[['customer_unique_id', 'segment_name']].to_csv("customer_segments.csv", index=False, encoding='utf-8-sig')

print("✅ تم حفظ الملف بنجاح بصيغة UTF-8: customer_segments2.csv")


✅ تم حفظ الملف بنجاح بصيغة UTF-8: customer_segments2.csv


In [30]:
avg_order_value.to_csv("customer_segments_value1.csv", index=False, encoding='utf-8-sig')

print("✅ تم حفظ الملف بنجاح بصيغة UTF-8: customer_segments_value.csv")


✅ تم حفظ الملف بنجاح بصيغة UTF-8: customer_segments_value.csv


In [31]:
cluster_summary = features.copy()
cluster_summary['cluster'] = kmeans.labels_
cluster_summary.groupby('cluster').mean()


,num_orders,total_spent,avg_order_value
cluster,,,
0,1.000000,109.663321,109.663321
1,1.021789,1736.339897,1709.945149
2,1.000000,493.046809,493.046809
3,2.113906,294.711976,139.307503
